In [8]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time
import os

In [6]:
def getDateofNewsfromCSV(save_path, symbol):
    if os.path.exists(save_path):
        df = pd.read_csv(f"{save_path}")
        print(f"{symbol}news.csv file exist!")
        if not df.empty:
            df["Date"] = pd.to_datetime(df["Date"])
            save_date_ts = df["Date"].iloc[0]
            save_date = save_date_ts.date()
            return save_date, df
            
        else:
            return None, df
    else:
        print(f"{symbol}news.csv file not exist!")
        return None, None

def fetchFullContent(link, date, headline, data, symbol, i):
    print(f"{(i+1)} ✓ Fetching {symbol}...", end=" ")
    try:
        response = requests.get(link, timeout=10)
        response.raise_for_status()
        article_soup = BeautifulSoup(response.content, "html.parser")
        content_div = article_soup.find("div", id="newsdetail-content")
        content = content_div.get_text(separator="\n", strip=True) if content_div else ""
    except Exception as e:
        print(f"Failed to fetch {link} for {headline}: {e}")
        content = ""

    data["Date"].append(date)
    data["Headline"].append(headline)
    data["Link"].append(link)
    data["Full Content"].append(content)
    print(f"✓ Data append...", end="   ")
    return 

query = "NEPSECompanyExtractor"
companyDetails = pd.read_csv(f"../DATA-HTML-STOCK/NEPSECompany/{query}.csv")
symbols = companyDetails['Symbol']



for index, symbol in enumerate(symbols[3:]):
    
    save_date = None
    data = {'Date': [], 'Headline': [], 'Link': [], 'Full Content': []}
    
    html_path = f"../DATA-HTML-STOCK/webScrapped-htmlfiles/news-WEB-SCRAP-htmlfile/{symbol}news.html"
    save_path = f"../DATA-HTML-STOCK/NEPSENEWS/{symbol}news.csv"
    
    save_date, df = getDateofNewsfromCSV(save_path, symbol)
    print(f"✓ Parsing HTML {symbol}...\n")
    if os.path.exists(html_path):
        with open(html_path, encoding="utf-8") as f:
                soup = BeautifulSoup(f.read(), "html.parser")
    else:
        print(f"{symbol}news.html file not exist")
        continue


    soupbdy = soup.find("tbody")
    if soupbdy is None:
        print(f"No news table found for {symbol}")
        continue

    rows = soupbdy.find_all("tr")

    for i, row in enumerate(rows):
        
        cols = row.find_all("td")
        date = pd.to_datetime(cols[0].get_text(strip=True)).date()
        headline = cols[1].get_text(strip=True)
        link = cols[1].find("a")["href"].strip()

        
        if save_date is not None:
            if date<=save_date:
                break

        fetchFullContent(link, date, headline, data, symbol, i)
        time.sleep(1)

    new_df = pd.DataFrame(data)
    if save_date is not None:
        if not new_df.empty:
            final_df = pd.concat([new_df, df], ignore_index=True)
            final_df.to_csv(save_path, index=False)
        else:
            print(f"No new news for {symbol}")
    else:       
        new_df.to_csv(save_path, index=False)  
    print(f"✓ Parsing HTML {symbol}... Done")
    print(f"✓ {(len(symbols)-index)} remain to parse...\n")
    
print("All Done!")

HATHPOnews.csv file not exist!
✓ Parsing HTML HATHPO...

HATHPOnews.html file not exist
AKPLnews.csv file not exist!
✓ Parsing HTML AKPL...

1 ✓ Fetching AKPL... ✓ Data append...   2 ✓ Fetching AKPL... ✓ Data append...   3 ✓ Fetching AKPL... ✓ Data append...   4 ✓ Fetching AKPL... ✓ Data append...   5 ✓ Fetching AKPL... ✓ Data append...   6 ✓ Fetching AKPL... ✓ Data append...   7 ✓ Fetching AKPL... ✓ Data append...   8 ✓ Fetching AKPL... ✓ Data append...   9 ✓ Fetching AKPL... ✓ Data append...   10 ✓ Fetching AKPL... ✓ Data append...   11 ✓ Fetching AKPL... ✓ Data append...   12 ✓ Fetching AKPL... ✓ Data append...   13 ✓ Fetching AKPL... ✓ Data append...   14 ✓ Fetching AKPL... ✓ Data append...   15 ✓ Fetching AKPL... ✓ Data append...   16 ✓ Fetching AKPL... ✓ Data append...   17 ✓ Fetching AKPL... ✓ Data append...   18 ✓ Fetching AKPL... ✓ Data append...   19 ✓ Fetching AKPL... ✓ Data append...   20 ✓ Fetching AKPL... ✓ Data append...   21 ✓ Fetching AKPL... ✓ Data append...   22 ✓ Fe

NotImplementedError: date not yet supported on Timestamps which are outside the range of Python's standard library. 

In [7]:
def getDateofNewsfromCSV(save_path, symbol):
    if os.path.exists(save_path):
        df = pd.read_csv(save_path)
        print(f"{symbol}news.csv file exists!")

        if not df.empty:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df = df.dropna(subset=["Date"])

            if not df.empty:
                save_date = df["Date"].iloc[0]
                return save_date, df

        return None, df

    else:
        print(f"{symbol}news.csv file not exist!")
        return None, None


def fetchFullContent(link, date, headline, data, symbol, i):
    print(f"{i+1} ✓ Fetching {symbol}...", end=" ")

    try:
        response = requests.get(link, timeout=10)
        response.raise_for_status()
        article_soup = BeautifulSoup(response.content, "html.parser")

        content_div = article_soup.find("div", id="newsdetail-content")
        content = (
            content_div.get_text(separator="\n", strip=True)
            if content_div else ""
        )

    except Exception as e:
        print(f"Failed to fetch {link}: {e}")
        content = ""

    data["Date"].append(date)
    data["Headline"].append(headline)
    data["Link"].append(link)
    data["Full Content"].append(content)

    print("✓ Data appended")
    return



query = "NEPSECompanyExtractor"
companyDetails = pd.read_csv(f"../DATA-HTML-STOCK/NEPSECompany/{query}.csv")
symbols = companyDetails["Symbol"]
stock_ignored = [ "CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "LFCPO", "SBIPO", "AMFIPO", "GABLPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]
symbol_s = ["GRDBL", "GMFIL"]
# GRDBL GMFIL

for index, symbol in enumerate(symbol_s):

    if symbol in stock_ignored:
        print(f"\n{symbol} is in ignore list!\n")
        continue

    data = {"Date": [],"Headline": [],"Link": [],"Full Content": []}

    html_path = f"../DATA-HTML-STOCK/webScrapped-htmlfiles/news-WEB-SCRAP-htmlfile/{symbol}news.html" 
    save_path = f"../DATA-HTML-STOCK/NEPSENEWS/{symbol}news.csv"

    save_date, old_df = getDateofNewsfromCSV(save_path, symbol)

    print(f"\n✓ Parsing HTML {symbol}...")

    if not os.path.exists(html_path):
        print(f"{symbol}news.html file not exist")
        continue

    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    soupbdy = soup.find("tbody")

    if soupbdy is None:
        print(f"No news table found for {symbol}")
        continue

    rows = soupbdy.find_all("tr")

    for i, row in enumerate(rows):

        cols = row.find_all("td")

        if len(cols) < 2:
            continue

        date_text = cols[0].get_text(strip=True)
        parsed_date = pd.to_datetime(date_text, errors="coerce")

        if pd.isna(parsed_date):
            print(f"⚠ Skipping invalid date: {date_text}")
            continue

        date = parsed_date

        headline = cols[1].get_text(strip=True)

        link_tag = cols[1].find("a")
        if not link_tag or not link_tag.get("href"):
            continue

        link = link_tag["href"].strip()

        if save_date is not None:
            if date <= save_date:
                break

        fetchFullContent(link, date, headline, data, symbol, i)
        time.sleep(1)

    new_df = pd.DataFrame(data)

    if save_date is not None:

        if not new_df.empty:
            final_df = pd.concat(
                [new_df, old_df],
                ignore_index=True
            )
            final_df = final_df.sort_values(
                by="Date", ascending=False
            )
            final_df.to_csv(save_path, index=False)
            print(f"✓ Updated {symbol}news.csv")

        else:
            print(f"No new news for {symbol}")

    else:
        if not new_df.empty:
            new_df = new_df.sort_values(
                by="Date", ascending=False
            )
            new_df.to_csv(save_path, index=False)
            print(f"✓ Created {symbol}news.csv")

    print(f"✓ {len(symbols) - index} remain to parse...\n")


print("All Done!")

CFCLnews.csv file exists!

✓ Parsing HTML CFCL...
No new news for CFCL
✓ 300 remain to parse...

CFCLPOnews.csv file not exist!

✓ Parsing HTML CFCLPO...
CFCLPOnews.html file not exist
CCBLnews.csv file not exist!

✓ Parsing HTML CCBL...
1 ✓ Fetching CCBL... ✓ Data appended
2 ✓ Fetching CCBL... ✓ Data appended
3 ✓ Fetching CCBL... ✓ Data appended
4 ✓ Fetching CCBL... ✓ Data appended
5 ✓ Fetching CCBL... ✓ Data appended
6 ✓ Fetching CCBL... ✓ Data appended
7 ✓ Fetching CCBL... ✓ Data appended
8 ✓ Fetching CCBL... ✓ Data appended
9 ✓ Fetching CCBL... ✓ Data appended
10 ✓ Fetching CCBL... ✓ Data appended
11 ✓ Fetching CCBL... ✓ Data appended
12 ✓ Fetching CCBL... ✓ Data appended
13 ✓ Fetching CCBL... ✓ Data appended
14 ✓ Fetching CCBL... ✓ Data appended
15 ✓ Fetching CCBL... ✓ Data appended
16 ✓ Fetching CCBL... ✓ Data appended
17 ✓ Fetching CCBL... ✓ Data appended
18 ✓ Fetching CCBL... ✓ Data appended
19 ✓ Fetching CCBL... ✓ Data appended
20 ✓ Fetching CCBL... ✓ Data appended
21 ✓ Fetchi

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



✓ Data appended
174 ✓ Fetching NTC... ✓ Data appended
175 ✓ Fetching NTC... ✓ Data appended
176 ✓ Fetching NTC... ✓ Data appended
177 ✓ Fetching NTC... ✓ Data appended
178 ✓ Fetching NTC... ✓ Data appended
179 ✓ Fetching NTC... ✓ Data appended
180 ✓ Fetching NTC... ✓ Data appended
181 ✓ Fetching NTC... ✓ Data appended
182 ✓ Fetching NTC... ✓ Data appended
183 ✓ Fetching NTC... ✓ Data appended
184 ✓ Fetching NTC... ✓ Data appended
185 ✓ Fetching NTC... ✓ Data appended
186 ✓ Fetching NTC... ✓ Data appended
187 ✓ Fetching NTC... ✓ Data appended
188 ✓ Fetching NTC... ✓ Data appended
189 ✓ Fetching NTC... ✓ Data appended
190 ✓ Fetching NTC... ✓ Data appended
191 ✓ Fetching NTC... ✓ Data appended
192 ✓ Fetching NTC... ✓ Data appended
193 ✓ Fetching NTC... ✓ Data appended
194 ✓ Fetching NTC... ✓ Data appended
195 ✓ Fetching NTC... ✓ Data appended
196 ✓ Fetching NTC... ✓ Data appended
197 ✓ Fetching NTC... ✓ Data appended
198 ✓ Fetching NTC... ✓ Data appended
199 ✓ Fetching NTC... ✓ Data appen

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



✓ Data appended
172 ✓ Fetching NICA... ✓ Data appended
173 ✓ Fetching NICA... ✓ Data appended
174 ✓ Fetching NICA... ✓ Data appended
175 ✓ Fetching NICA... ✓ Data appended
176 ✓ Fetching NICA... ✓ Data appended
177 ✓ Fetching NICA... ✓ Data appended
178 ✓ Fetching NICA... ✓ Data appended
179 ✓ Fetching NICA... ✓ Data appended
180 ✓ Fetching NICA... ✓ Data appended
181 ✓ Fetching NICA... ✓ Data appended
182 ✓ Fetching NICA... ✓ Data appended
183 ✓ Fetching NICA... ✓ Data appended
184 ✓ Fetching NICA... ✓ Data appended
185 ✓ Fetching NICA... ✓ Data appended
186 ✓ Fetching NICA... ✓ Data appended
187 ✓ Fetching NICA... ✓ Data appended
188 ✓ Fetching NICA... ✓ Data appended
189 ✓ Fetching NICA... ✓ Data appended
190 ✓ Fetching NICA... ✓ Data appended
191 ✓ Fetching NICA... ✓ Data appended
192 ✓ Fetching NICA... ✓ Data appended
193 ✓ Fetching NICA... ✓ Data appended
194 ✓ Fetching NICA... ✓ Data appended
195 ✓ Fetching NICA... ✓ Data appended
196 ✓ Fetching NICA... ✓ Data appended
197 ✓ Fet

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



✓ Data appended
43 ✓ Fetching SHINE... ✓ Data appended
44 ✓ Fetching SHINE... ✓ Data appended
45 ✓ Fetching SHINE... ✓ Data appended
46 ✓ Fetching SHINE... ✓ Data appended
47 ✓ Fetching SHINE... ✓ Data appended
48 ✓ Fetching SHINE... ✓ Data appended
49 ✓ Fetching SHINE... ✓ Data appended
50 ✓ Fetching SHINE... ✓ Data appended
51 ✓ Fetching SHINE... ✓ Data appended
52 ✓ Fetching SHINE... ✓ Data appended
53 ✓ Fetching SHINE... ✓ Data appended
54 ✓ Fetching SHINE... ✓ Data appended
55 ✓ Fetching SHINE... ✓ Data appended
56 ✓ Fetching SHINE... ✓ Data appended
57 ✓ Fetching SHINE... ✓ Data appended
58 ✓ Fetching SHINE... ✓ Data appended
59 ✓ Fetching SHINE... ✓ Data appended
60 ✓ Fetching SHINE... ✓ Data appended
61 ✓ Fetching SHINE... ✓ Data appended
62 ✓ Fetching SHINE... ✓ Data appended
63 ✓ Fetching SHINE... ✓ Data appended
64 ✓ Fetching SHINE... ✓ Data appended
65 ✓ Fetching SHINE... ✓ Data appended
66 ✓ Fetching SHINE... ✓ Data appended
67 ✓ Fetching SHINE... ✓ Data appended
68 ✓ Fetc

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



104 ✓ Fetching TRH... ✓ Data appended
105 ✓ Fetching TRH... ✓ Data appended
✓ Created TRHnews.csv
✓ 58 remain to parse...

TNBLnews.csv file not exist!

✓ Parsing HTML TNBL...
1 ✓ Fetching TNBL... ✓ Data appended
2 ✓ Fetching TNBL... ✓ Data appended
3 ✓ Fetching TNBL... ✓ Data appended
4 ✓ Fetching TNBL... ✓ Data appended
5 ✓ Fetching TNBL... ✓ Data appended
6 ✓ Fetching TNBL... ✓ Data appended
7 ✓ Fetching TNBL... ✓ Data appended
8 ✓ Fetching TNBL... ✓ Data appended
9 ✓ Fetching TNBL... ✓ Data appended
10 ✓ Fetching TNBL... ✓ Data appended
11 ✓ Fetching TNBL... ✓ Data appended
12 ✓ Fetching TNBL... ✓ Data appended
13 ✓ Fetching TNBL... ✓ Data appended
14 ✓ Fetching TNBL... ✓ Data appended
15 ✓ Fetching TNBL... ✓ Data appended
16 ✓ Fetching TNBL... ✓ Data appended
17 ✓ Fetching TNBL... ✓ Data appended
18 ✓ Fetching TNBL... ✓ Data appended
19 ✓ Fetching TNBL... ✓ Data appended
20 ✓ Fetching TNBL... ✓ Data appended
21 ✓ Fetching TNBL... ✓ Data appended
22 ✓ Fetching TNBL... ✓ Data append

In [9]:
def getDateofNewsfromCSV(save_path, symbol):
    if os.path.exists(save_path):
        df = pd.read_csv(save_path)
        print(f"{symbol}news.csv file exists!")

        if not df.empty:
            df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            df = df.dropna(subset=["Date"])

            if not df.empty:
                save_date = df["Date"].iloc[0]
                return save_date, df

        return None, df

    else:
        print(f"{symbol}news.csv file not exist!")
        return None, None


def fetchFullContent(link, date, headline, data, symbol, i):
    print(f"{i+1} ✓ Fetching {symbol}...", end=" ")

    try:
        response = requests.get(link, timeout=10)
        response.raise_for_status()
        article_soup = BeautifulSoup(response.content, "html.parser")

        content_div = article_soup.find("div", id="newsdetail-content")
        content = (
            content_div.get_text(separator="\n", strip=True)
            if content_div else ""
        )

    except Exception as e:
        print(f"Failed to fetch {link}: {e}")
        content = ""

    data["Date"].append(date)
    data["Headline"].append(headline)
    data["Link"].append(link)
    data["Full Content"].append(content)

    print("✓ Data appended")
    return



query = "NEPSECompanyExtractor"
companyDetails = pd.read_csv(f"../DATA-HTML-STOCK/NEPSECompany/{query}.csv")
symbols = companyDetails["Symbol"]
stock_ignored = [ "CORBLP", "GFCLPO", "HBLPO", "LFCPO", "SBIPO", "GABLPO", "LFCPO", "SBIPO", "AMFIPO", "GABLPO"]
stock_no_news = ["HATHPO", "BUDBLP", "CFCLPO", "CORBLP", "EBLPO", "HAMROP", "HGIPO", "JFLPO", "JBNLPO", "KADBLP", "KNBLPO", "CEFLPO", "LBLPO", "MFILPO", "MIDBLP", "NBBLPO", "NABBPO", "NABBPO", "NBBPO", "PFLPO", "PRINPO", "PURBLP", "SBBLJP", "SIFCPO", "SILPO", "SMFDBP", "SYFLPO", "TNBLPO", "TDBLPO", "UFLPO", "WDBLPO", "WMBFPO", "HLBSLP"]
symbol_s = ["GRDBL", "GMFIL"]

for index, symbol in enumerate(symbol_s):

    if symbol in stock_ignored:
        print(f"\n{symbol} is in ignore list!\n")
        continue

    data = {"Date": [],"Headline": [],"Link": [],"Full Content": []}

    html_path = f"../DATA-HTML-STOCK/webScrapped-htmlfiles/news-WEB-SCRAP-htmlfile/{symbol}news.html" 
    save_path = f"../DATA-HTML-STOCK/NEPSENEWS/{symbol}news.csv"

    save_date, old_df = getDateofNewsfromCSV(save_path, symbol)

    print(f"\n✓ Parsing HTML {symbol}...")

    if not os.path.exists(html_path):
        print(f"{symbol}news.html file not exist")
        continue

    with open(html_path, encoding="utf-8") as f:
        soup = BeautifulSoup(f.read(), "html.parser")

    soupbdy = soup.find("tbody")

    if soupbdy is None:
        print(f"No news table found for {symbol}")
        continue

    rows = soupbdy.find_all("tr")

    for i, row in enumerate(rows):

        cols = row.find_all("td")

        if len(cols) < 2:
            continue

        date_text = cols[0].get_text(strip=True)
        parsed_date = pd.to_datetime(date_text, errors="coerce")

        if pd.isna(parsed_date):
            print(f"⚠ Skipping invalid date: {date_text}")
            continue

        date = parsed_date

        headline = cols[1].get_text(strip=True)

        link_tag = cols[1].find("a")
        if not link_tag or not link_tag.get("href"):
            continue

        link = link_tag["href"].strip()

        if save_date is not None:
            if date <= save_date:
                break

        fetchFullContent(link, date, headline, data, symbol, i)
        time.sleep(1)

    new_df = pd.DataFrame(data)

    if save_date is not None:

        if not new_df.empty:
            final_df = pd.concat(
                [new_df, old_df],
                ignore_index=True
            )
            final_df = final_df.sort_values(
                by="Date", ascending=False
            )
            final_df.to_csv(save_path, index=False)
            print(f"✓ Updated {symbol}news.csv")

        else:
            print(f"No new news for {symbol}")

    else:
        if not new_df.empty:
            new_df = new_df.sort_values(
                by="Date", ascending=False
            )
            new_df.to_csv(save_path, index=False)
            print(f"✓ Created {symbol}news.csv")

    print(f"✓ {len(symbols) - index} remain to parse...\n")


print("All Done!")

GRDBLnews.csv file not exist!

✓ Parsing HTML GRDBL...
GRDBLnews.html file not exist
GMFILnews.csv file not exist!

✓ Parsing HTML GMFIL...
GMFILnews.html file not exist
All Done!
